# Direct TS Prediction

Instead of searching along a path, we generate TS candidates directly using an AI model.

## How does it work?

tsdiff uses a diffusion model (hence the name). Diffusion models are very well suited for this task, because multiple transition states are often close to each other in energy even though they sometimes belong to different reaction classes, e.g. substitution vs. elimination. The diffusion model will start from a random guess, which it prints out and is very far away from any sensible chemical structure, and in each iteration improves the structures through denoising it. Think about it as the network actively diffusing the input towards the output (again hence the name). On top of that tsdiff does a good job of identifying reaction-patterns, by mapping the molecule onto a 2D graph.

Now we are testing it for a proton transfer between hydronium and water as well as a proton transfer between CH4 and CH3.

If you have a GPU I once again highly recommend using it, by setting the pytorch device.

In [ ]:
# create train, validation and test set, but in this case
# we are only interested in testing custom cases, so only
# the test set will contain data.

from helpers.preprocessing_helper import preprocess_args, preprocess
from helpers.sampling_helper import sampling_args, sample

In [ ]:
# class attributes contain parameters
prep_params = preprocess_args()
prep_params.train = 0.0
prep_params.valid = 0.0
# generate preprocessed files
preprocess(prep_params)

In [ ]:
# run a pretrained model on the generated test data
sample_params = sampling_args()
#sample_params.device = "cuda"
# only use one model for faster calculation
# if you want to use all 8 available ones, just uncomment this line
sample_params.ckpt = [sample_params.ckpt[0]]
sample_params.n_steps = 5000
sample(sample_params)

In [ ]:
# visualize the generated transition state structures
import py3Dmol

def view_xyz(filename):
    with open(filename, "r") as f:
        xyz = f.read()

        viewer = py3Dmol.view(width=500, height=400)
        viewer.addModel(xyz, "xyz")

        # nice default style
        viewer.setStyle({
            "stick": {"radius": 0.15},
            "sphere": {"scale": 0.3}
        })

        viewer.zoomTo()
    return viewer

view_xyz("ts0.xyz")

In [ ]:
view_xyz("ts1.xyz")

## Clustering and Training

A nice feature of the diffusion methodology is that when running this multiple times, different transition state structures will be found, if the PES is reasonably complex. If that is the case you can use tsdiffs clustering tool, which groups the reactions e.g. into competing reaction channels. Our test reactions are, however, too simple for that.

Regarding training there is, to the best of my knowledge, no short retraining opportunity for tsdiff, so if you want to add something you have to fully retrain from your seed.

If you are interested you can find the GitHub repository for tsdiff [here](https://github.com/seonghann/tsdiff).

## Where Do TS Predictors Work Well?

| System Type | Performance |
|------------|------------|
| Small organic reactions | Excellent |
| Proton transfer | Very strong |
| SN2 reactions | Strong |
| Rearrangements | Mixed |
| Large flexible systems | Challenging |
| Transition metals | Poor |
| Charged/solvent systems | Difficult |

Conclusion:
AI TS predictors are powerful but require hybrid workflows and validation.

# Now try tsdiff for your own reaction

Now for the fun part, where you create your own reaction and can use tsdiff to predict the transition state structure.

Therefore you will have to do the following:
- Create SMARTS for the reactions you are interested in, e.g. with ChatGPT
- Create xyz structure of reactants/products/whatever (this is just for workflow consistency with tsdiff)
- Create your new "test set" with the preprocessing routine
- Generate the transition state structure using the sampling routine

You can just append smarts.csv and structures.xyz in data/raw_data/ with the new smarts and xyz, but **beware of the atom ordering** in the smarts and the xyz! The atoms in the smarts need to have the same enumeration for reactants and products, and this ordering also needs to apply for the xyz coordinates. To give an example, if the reactant side of the smart has an O at position 2 and H at position three, but it's vice versa on the product side, this won't work.

In order to (only) calculate your new test reaction, you also need to adjust end_idx from 2 to 3.